# Tugas 1 – Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Tegar Reski Pratama  
**NIM:** 202310370311215  
**Username GitHub:** TegarReskiPratama

Notebook mengikuti empat bagian template `03_eda_and_insights.ipynb` yang diberikan.
Dataset: **Indonesian News Articles Published at 2017**, hasil cleaning Milestone 2.
Unit analisis adalah artikel dalam arsip, bukan peristiwa atau sensus berita Indonesia.
Dipilih analisis temporal dan multidimensi karena tidak tersedia koordinat terverifikasi.

**Pemenuhan template:** pemuatan Polars, window function DuckDB `OVER`, 10 grafik
interaktif Plotly (enam jenis wajib dan empat pendalaman), serta sembilan insight
yang dihitung saat eksekusi. Semua agregasi memakai seluruh korpus bersih.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Polars membaca hanya kolom analisis dari Parquet hasil Milestone 2 sehingga isi
artikel tidak perlu dimuat lagi. Checksum dan jumlah baris harus cocok dengan
`cleaning_summary.json`. DuckDB kemudian membaca Parquet untuk agregasi dan window.

In [1]:
from pathlib import Path
import math
import platform
import sys

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / 'requirements.txt').is_file() and (path / 'data/README.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Buka Jupyter dari akar repository atau folder notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import numpy as np
import pandas as pd
import polars as pl
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from src.common import (
    connect_analytics, prepare_directories, read_json, sha256_file, sql_literal, write_json,
)

prepare_directories(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data/processed/dataset_bersih.parquet'
FIGURE_DIR = PROJECT_ROOT / 'output/figures'
CLEANING = read_json(PROJECT_ROOT / 'output/cleaning_summary.json')
if not DATA_PATH.is_file():
    raise FileNotFoundError('Jalankan notebook 02 terlebih dahulu.')
assert sha256_file(DATA_PATH) == CLEANING['clean_parquet_sha256'], 'Data bersih berubah; ulangi notebook 02.'
assert CLEANING['rows_raw'] == (CLEANING['rows_clean'] + CLEANING['rows_rejected']
                                + CLEANING['duplicates_removed'])

analysis_columns = [
    'row_id', 'published_at', 'published_date', 'month', 'weekday', 'hour', 'has_time',
    'source', 'word_count', 'char_count', 'is_length_outlier', 'source_is_imputed',
]
df = pl.scan_parquet(DATA_PATH).select(analysis_columns).collect(engine='streaming')
total = df.height
assert total == CLEANING['rows_clean'] and total > 0
assert df.select(pl.col('published_date').dt.year().ne(2017).any()).item() is False
print(f'Data Polars: {total:,} artikel × {df.width} kolom analisis')
display(df.head(5))
del df  # Agregasi berikutnya membaca Parquet, tanpa mempertahankan salinan di memori.

connection = connect_analytics(PROJECT_ROOT)
connection.execute(f"CREATE VIEW news AS SELECT {', '.join(analysis_columns)} "
                   f"FROM read_parquet({sql_literal(DATA_PATH)})")
pio.templates.default = 'plotly_white'
pio.renderers.default = 'plotly_mimetype'
DAY_NAMES = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
figures = {}


def show_and_save(figure, name):
    figure.update_layout(font=dict(family='Arial', size=13), hovermode='closest')
    figure.write_html(FIGURE_DIR / f'{name}.html', include_plotlyjs=True, full_html=True)
    figures[name] = figure.layout.title.text
    figure.show()

Data Polars: 455,360 artikel × 12 kolom analisis


row_id,published_at,published_date,month,weekday,hour,has_time,source,word_count,char_count,is_length_outlier,source_is_imputed
u32,datetime[μs],date,date,i8,i8,bool,str,u32,u32,bool,bool
0,2017-01-01 00:00:00,2017-01-01,2017-01-01,7,0,true,"""liputan6.com""",347,2561,false,false
1,2017-01-01 00:01:00,2017-01-01,2017-01-01,7,0,true,"""liputan6.com""",913,6473,true,false
2,2017-01-01 00:03:09,2017-01-01,2017-01-01,7,0,true,"""kompas.com""",126,913,false,false
3,2017-01-01 00:04:21,2017-01-01,2017-01-01,7,0,true,"""liputan6.com""",163,1147,false,false
4,2017-01-01 00:05:08,2017-01-01,2017-01-01,7,0,true,"""liputan6.com""",286,1978,false,false


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Kalender lengkap 1 Januari–31 Desember 2017 di-left join dengan hitungan artikel.
`AVG(...) OVER (ORDER BY tanggal ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`
menghitung rata-rata bergerak tujuh **hari kalender**, termasuk nol bila arsip kosong.
Enam hari pertama memakai jumlah hari tersedia (1–6 hari), ditunjukkan pada hover.

Rata-rata panjang dan proporsi outlier dihitung per hari untuk korelasi pada Bagian 3.
Hari tanpa artikel tetap nol untuk volume, tetapi statistik panjang tetap null;
hari tersebut dikeluarkan dari korelasi, bukan diisi dengan panjang nol.

In [2]:
connection.execute("""
    CREATE TEMP TABLE calendar AS
    SELECT CAST(day AS DATE) AS tanggal,
           CAST(DATE_TRUNC('month', day) AS DATE) AS bulan,
           CAST(ISODOW(day) AS INTEGER) AS weekday
    FROM GENERATE_SERIES(DATE '2017-01-01', DATE '2017-12-31', INTERVAL 1 DAY) AS t(day)
""")
connection.execute("""
    CREATE TEMP TABLE daily_counts AS
    SELECT c.tanggal, c.bulan, c.weekday, COALESCE(n.artikel, 0) AS artikel,
           n.artikel IS NOT NULL AS ada_artikel, n.mean_words, n.outlier_percent
    FROM calendar c
    LEFT JOIN (
        SELECT published_date AS tanggal, COUNT(*) AS artikel,
               AVG(word_count) AS mean_words,
               100.0 * COUNT(*) FILTER (WHERE is_length_outlier) / COUNT(*) AS outlier_percent
        FROM news GROUP BY published_date
    ) n USING (tanggal)
""")
daily = connection.execute("""
    SELECT *, AVG(artikel) OVER w AS ma7, COUNT(*) OVER w AS window_days
    FROM daily_counts
    WINDOW w AS (ORDER BY tanggal ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
    ORDER BY tanggal
""").df()
assert len(daily) == 365 and int(daily['artikel'].sum()) == total
assert daily['tanggal'].is_unique
assert np.allclose(daily['ma7'], daily['artikel'].rolling(7, min_periods=1).mean())
daily.to_csv(PROJECT_ROOT / 'output/eda_daily.csv', index=False)
display(daily.head(8))

,tanggal,bulan,weekday,artikel,ada_artikel,mean_words,outlier_percent,ma7,window_days
0,2017-01-01,2017-01-01,7,749,True,232.365821,3.738318,749.000000,1
1,2017-01-02,2017-01-01,1,1322,True,255.481089,4.614221,1035.500000,2
2,2017-01-03,2017-01-01,2,1457,True,253.555937,3.980782,1176.000000,3
3,2017-01-04,2017-01-01,3,1572,True,265.050254,4.452926,1275.000000,4
4,2017-01-05,2017-01-01,4,1635,True,266.684404,4.892966,1347.000000,5
5,2017-01-06,2017-01-01,5,1498,True,266.479306,4.539386,1372.166667,6
6,2017-01-07,2017-01-01,6,866,True,249.657044,2.886836,1299.857143,7
7,2017-01-08,2017-01-01,7,870,True,257.895402,4.022989,1317.142857,7


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Enam grafik pertama mengikuti urutan jenis visualisasi template. Empat grafik
tambahan mendalami volume bulanan, pola hari, perubahan sumber, dan panjang teks.
Semua grafik mendukung hover, zoom, dan ekspor; legenda seri dapat diklik.
HTML mandiri disimpan di `output/figures/` untuk dibuka secara offline.

### Visualisasi 1: Tren Deret Waktu dengan Range Slider

Bandingkan volume harian dengan rata-rata tujuh hari untuk memisahkan fluktuasi
jangka pendek dari pergerakan mingguan. Nol berarti tidak ada artikel pada arsip,
bukan bukti bahwa media tidak menerbitkan berita.

In [3]:
fig1 = go.Figure()
fig1.add_scatter(x=daily['tanggal'], y=daily['artikel'], name='Artikel harian',
                 hovertemplate='%{x|%d %b %Y}<br>Artikel: %{y:,}<extra></extra>')
fig1.add_scatter(x=daily['tanggal'], y=daily['ma7'], name='Rata-rata 7 hari',
                 customdata=daily[['window_days']], line=dict(width=3),
                 hovertemplate='%{x|%d %b %Y}<br>Rata-rata: %{y:.2f}'
                               '<br>Hari dalam window: %{customdata[0]}<extra></extra>')
fig1.update_layout(title='1. Jumlah Artikel Harian dan Rata-rata 7 Hari',
                   xaxis_title='Tanggal (2017)', yaxis_title='Artikel dalam dataset')
fig1.update_xaxes(rangeslider_visible=True)
show_and_save(fig1, '01_daily_trend')

### Visualisasi 2: Distribusi Variabel Utama dan Outlier

Histogram menggunakan bin 50 kata. Bin terakhir menggabungkan ekor di atas batas
pembulatan persentil 99; tidak ada artikel yang dibuang. Flag IQR berasal dari
artikel valid sebelum deduplikasi pada Milestone 2, dan tetap dipertahankan.

In [4]:
length = connection.execute("""
    SELECT AVG(word_count) AS mean_words, MEDIAN(word_count) AS median_words,
           QUANTILE_CONT(word_count, 0.99) AS p99,
           MIN(word_count) AS min_words, MAX(word_count) AS max_words,
           COUNT(*) FILTER (WHERE is_length_outlier) AS outliers
    FROM news
""").df().iloc[0]
tail_cutoff = max(100, int(math.ceil(float(length['p99']) / 50) * 50))
histogram = connection.execute("""
    SELECT LEAST(CAST(FLOOR(word_count / 50.0) * 50 AS INTEGER), ?) AS bin_start,
           COUNT(*) AS artikel
    FROM news GROUP BY bin_start ORDER BY bin_start
""", [tail_cutoff]).df()
histogram['rentang_kata'] = [
    f'{int(start)}–{int(start) + 49}' if start < tail_cutoff else f'≥{tail_cutoff}'
    for start in histogram['bin_start']
]
assert int(histogram['artikel'].sum()) == total
histogram.to_csv(PROJECT_ROOT / 'output/eda_length_histogram.csv', index=False)
fig2 = px.bar(histogram, x='rentang_kata', y='artikel',
              labels={'rentang_kata': 'Jumlah kata (ekor digabung di bin terakhir)', 'artikel': 'Artikel'},
              title='2. Distribusi Jumlah Kata per Artikel')
show_and_save(fig2, '02_article_length_distribution')

### Visualisasi 3: Korelasi Antarvariabel Numerik

Heatmap Pearson memakai satu observasi per hari yang memiliki artikel: volume,
rata-rata jumlah kata, dan persentase outlier. Tidak ada sampling artikel.
Koefisien berada pada −1 sampai 1; null berarti korelasi tidak dapat dihitung.
Hubungan ini bersifat deskriptif pada tingkat hari, bukan artikel, dan dapat
dipengaruhi komposisi sumber serta autokorelasi waktu. Tidak dilakukan uji kausal
atau klaim signifikansi statistik. Rata-rata panjang dan flag outlier sama-sama
diturunkan dari panjang teks sehingga hubungan keduanya perlu dibaca hati-hati.

In [5]:
correlation_columns = ['artikel', 'mean_words', 'outlier_percent']
correlation_labels = ['Artikel/hari', 'Rata-rata kata/hari', 'Outlier/hari (%)']
correlation_days = daily.loc[daily['ada_artikel'], correlation_columns].dropna()
correlation = correlation_days.corr(method='pearson')
correlation.to_csv(PROJECT_ROOT / 'output/eda_daily_correlation.csv', index_label='variabel')
volume_length_r = float(correlation.loc['artikel', 'mean_words'])
fig3 = px.imshow(correlation.to_numpy(), x=correlation_labels, y=correlation_labels,
                 text_auto='.3f', zmin=-1, zmax=1, color_continuous_midpoint=0,
                 color_continuous_scale='RdBu_r', aspect='auto',
                 labels={'color': 'Pearson r'},
                 title=f'3. Korelasi Metrik Harian ({len(correlation_days)} hari berartikel)')
fig3.update_traces(hovertemplate='%{y} vs %{x}<br>Pearson r: %{z:.3f}<extra></extra>')
show_and_save(fig3, '03_daily_correlation')

### Visualisasi 4: Komposisi Kategori Sumber

Sepuluh label sumber terbesar ditampilkan; persentase selalu memakai seluruh
korpus. Label/domain sumber tidak otomatis mewakili kategori topik berita.

In [6]:
sources = connection.execute("""
    SELECT source AS sumber, COUNT(*) AS artikel,
           AVG(word_count) AS rata_kata, MEDIAN(word_count) AS median_kata,
           100.0 * COUNT(*) FILTER (WHERE is_length_outlier) / COUNT(*) AS outlier_percent,
           COUNT(*) FILTER (WHERE source_is_imputed) AS sumber_imputasi
    FROM news GROUP BY source ORDER BY artikel DESC, sumber
""").df()
sources['persentase'] = sources['artikel'] / total * 100
assert int(sources['artikel'].sum()) == total
sources.to_csv(PROJECT_ROOT / 'output/eda_sources.csv', index=False)
top_sources = sources.head(10).sort_values(['artikel', 'sumber'])
fig4 = px.bar(top_sources, x='artikel', y='sumber', orientation='h',
              hover_data=['persentase', 'rata_kata', 'outlier_percent'],
              labels={'sumber': 'Sumber tercatat', 'artikel': 'Artikel'},
              title='4. Komposisi Sumber Artikel Terbesar')
show_and_save(fig4, '04_source_composition')

### Visualisasi 5: Siklus Mingguan dan Jam Publikasi

Warna menunjukkan artikel per kejadian hari kalender agar 53 Minggu tidak
dibandingkan mentah dengan 52 Senin. Hover juga menampilkan jumlah artikel asli.
Hanya timestamp dengan jam eksplisit dipakai. Offset dikonversi ke WIB pada
cleaning; timestamp tanpa offset diasumsikan WIB, belum dikonfirmasi ke penyedia.

In [7]:
weekday = connection.execute("""
    SELECT weekday, SUM(artikel) AS artikel, COUNT(*) AS hari_kalender,
           COUNT(*) FILTER (WHERE ada_artikel) AS hari_teramati,
           AVG(artikel) AS rata_per_hari
    FROM daily_counts GROUP BY weekday ORDER BY weekday
""").df()
weekday['hari'] = [DAY_NAMES[int(day) - 1] for day in weekday['weekday']]
weekday.to_csv(PROJECT_ROOT / 'output/eda_weekday.csv', index=False)
hourly = connection.execute("""
    SELECT weekday, hour AS jam, COUNT(*) AS artikel
    FROM news WHERE has_time AND hour IS NOT NULL
    GROUP BY weekday, hour ORDER BY weekday, hour
""").df()
hourly.to_csv(PROJECT_ROOT / 'output/eda_hourly.csv', index=False)
known_time_rows = int(hourly['artikel'].sum())
heat_counts = pd.DataFrame(0, index=DAY_NAMES, columns=list(range(24)))
for row in hourly.itertuples(index=False):
    heat_counts.loc[DAY_NAMES[int(row.weekday) - 1], int(row.jam)] = int(row.artikel)
day_denominators = weekday.set_index('hari')['hari_kalender'].reindex(DAY_NAMES)
heat_rate = heat_counts.div(day_denominators, axis=0)
assert int(heat_counts.to_numpy().sum()) == known_time_rows
fig5 = px.imshow(heat_rate, labels=dict(x='Jam tercatat / WIB', y='Hari', color='Artikel/hari'),
                 title=f'5. Siklus Hari × Jam ({known_time_rows:,} artikel dengan jam)',
                 aspect='auto', color_continuous_scale='Blues')
fig5.update_traces(customdata=heat_counts.to_numpy(),
                    hovertemplate='%{y}, jam %{x}:00<br>Artikel/hari: %{z:.2f}'
                                  '<br>Total artikel: %{customdata:,}<extra></extra>')
if known_time_rows == 0:
    fig5.add_annotation(text='Jam publikasi tidak tersedia; analisis jam tidak dapat dilakukan.',
                        xref='paper', yref='paper', x=0.5, y=0.5, showarrow=False)
show_and_save(fig5, '05_weekday_hour_heatmap')

### Visualisasi 6: Korelasi Multidimensi dan Perbandingan Segmen

Setiap bubble adalah satu sumber: sumbu X volume artikel, Y rata-rata panjang,
ukuran proporsi outlier IQR, dan warna label sumber. Hover mencantumkan median
serta pangsa korpus. Semua sumber digunakan. Ini perbandingan deskriptif segmen,
bukan bukti bahwa volume menyebabkan perubahan panjang. Tidak digunakan peta
karena dataset tidak memiliki koordinat terverifikasi.

In [8]:
fig6 = px.scatter(sources, x='artikel', y='rata_kata', size='outlier_percent',
                  color='sumber', hover_name='sumber', size_max=45,
                  hover_data=['median_kata', 'persentase', 'outlier_percent'],
                  labels={'artikel': 'Artikel per sumber', 'rata_kata': 'Rata-rata kata/artikel',
                          'outlier_percent': 'Outlier (%)', 'sumber': 'Sumber',
                          'median_kata': 'Median kata', 'persentase': 'Pangsa korpus (%)'},
                  title='6. Volume, Panjang Teks, dan Outlier Menurut Sumber')
show_and_save(fig6, '06_source_multidimensional')

### Visualisasi 7–10: Pendalaman Temporal dan Perbandingan Kelompok

Volume bulanan dilengkapi rata-rata per hari untuk menilai perbedaan panjang bulan.
Pola hari memakai denominator kalender. Tren sumber memakai lima label terbesar
sepanjang tahun, dan mencakup semua bulan walaupun suatu sumber tidak punya artikel.
Rata-rata/median bulanan menggunakan panjang asli tanpa membuang outlier.

In [9]:
monthly = connection.execute("""
    SELECT bulan, SUM(artikel) AS artikel, COUNT(*) AS hari_kalender,
           COUNT(*) FILTER (WHERE ada_artikel) AS hari_teramati,
           AVG(artikel) AS rata_per_hari
    FROM daily_counts GROUP BY bulan ORDER BY bulan
""").df()
monthly['label_bulan'] = pd.to_datetime(monthly['bulan']).dt.strftime('%Y-%m')
monthly['persentase'] = monthly['artikel'] / total * 100
assert len(monthly) == 12 and int(monthly['artikel'].sum()) == total
monthly.to_csv(PROJECT_ROOT / 'output/eda_monthly.csv', index=False)
fig7 = px.bar(monthly, x='label_bulan', y='artikel', color='rata_per_hari',
              hover_data=['hari_kalender', 'hari_teramati', 'persentase'],
              labels={'label_bulan': 'Bulan', 'artikel': 'Artikel', 'rata_per_hari': 'Artikel/hari'},
              title='7. Jumlah Artikel per Bulan', color_continuous_scale='Blues')
show_and_save(fig7, '07_monthly_volume')

In [10]:
fig8 = px.bar(weekday, x='hari', y='rata_per_hari',
              hover_data=['artikel', 'hari_kalender', 'hari_teramati'],
              category_orders={'hari': DAY_NAMES}, color='rata_per_hari',
              labels={'hari': 'Hari', 'rata_per_hari': 'Rata-rata artikel/hari kalender'},
              title='8. Intensitas Artikel Menurut Hari dalam Minggu', color_continuous_scale='Teal')
show_and_save(fig8, '08_weekday_pattern')

In [11]:
top_source_trend = connection.execute("""
    WITH top_sources AS (
        SELECT source FROM news GROUP BY source ORDER BY COUNT(*) DESC, source LIMIT 5
    ), months AS (SELECT DISTINCT bulan FROM calendar), counts AS (
        SELECT month AS bulan, source, COUNT(*) AS artikel
        FROM news WHERE source IN (SELECT source FROM top_sources) GROUP BY month, source
    )
    SELECT m.bulan, t.source AS sumber, COALESCE(c.artikel, 0) AS artikel
    FROM months m CROSS JOIN top_sources t
    LEFT JOIN counts c ON c.bulan = m.bulan AND c.source = t.source
    ORDER BY m.bulan, t.source
""").df()
top_source_trend['label_bulan'] = pd.to_datetime(top_source_trend['bulan']).dt.strftime('%Y-%m')
top_source_trend.to_csv(PROJECT_ROOT / 'output/eda_source_monthly.csv', index=False)
fig9 = px.line(top_source_trend, x='label_bulan', y='artikel', color='sumber', markers=True,
               labels={'label_bulan': 'Bulan', 'artikel': 'Artikel', 'sumber': 'Sumber'},
               title='9. Volume Bulanan Lima Sumber Terbesar')
show_and_save(fig9, '09_source_monthly_trend')

In [12]:
length_monthly = connection.execute("""
    SELECT month AS bulan, COUNT(*) AS artikel, AVG(word_count) AS mean_words,
           MEDIAN(word_count) AS median_words,
           100.0 * COUNT(*) FILTER (WHERE is_length_outlier) / COUNT(*) AS outlier_percent
    FROM news GROUP BY month ORDER BY month
""").df()
length_monthly['label_bulan'] = pd.to_datetime(length_monthly['bulan']).dt.strftime('%Y-%m')
length_monthly.to_csv(PROJECT_ROOT / 'output/eda_monthly_length.csv', index=False)
length_long = length_monthly.melt(id_vars=['label_bulan', 'artikel', 'outlier_percent'],
                                 value_vars=['mean_words', 'median_words'],
                                 var_name='statistik', value_name='jumlah_kata')
fig10 = px.line(length_long, x='label_bulan', y='jumlah_kata', color='statistik', markers=True,
                hover_data=['artikel', 'outlier_percent'],
                labels={'label_bulan': 'Bulan', 'jumlah_kata': 'Kata/artikel', 'statistik': 'Statistik'},
                title='10. Rata-rata dan Median Panjang Artikel per Bulan')
show_and_save(fig10, '10_monthly_article_length')

## Bagian 4: Rumusan 5 Insight Analitik Utama

Sembilan temuan berikut mencakup lima kategori template: tren/musiman,
distribusi/anomali, korelasi, segmen sumber, dan rekomendasi praktis. Teks serta
angka dihasilkan dari tabel agregasi di atas dan disimpan sebagai laporan Markdown
serta ringkasan JSON. Tidak ada angka temuan yang ditulis berdasarkan tebakan.

In [13]:
most_month = monthly.sort_values(['artikel', 'label_bulan'], ascending=[False, True]).iloc[0]
least_month = monthly.sort_values(['artikel', 'label_bulan']).iloc[0]
busiest_day = weekday.sort_values(['rata_per_hari', 'weekday'], ascending=[False, True]).iloc[0]
top5_share = float(sources.head(5)['persentase'].sum())
top_source = sources.iloc[0]
shortest_source = sources.sort_values(['rata_kata', 'sumber']).iloc[0]
longest_source = sources.sort_values(['rata_kata', 'sumber'], ascending=[False, True]).iloc[0]
weekdays = weekday[weekday['weekday'] <= 5]
weekends = weekday[weekday['weekday'] >= 6]
weekday_rate = float(weekdays['artikel'].sum() / weekdays['hari_kalender'].sum())
weekend_rate = float(weekends['artikel'].sum() / weekends['hari_kalender'].sum())
coverage_days = int(daily['ada_artikel'].sum())
empty_days = len(daily) - coverage_days

if known_time_rows:
    by_hour = hourly.groupby('jam', as_index=False)['artikel'].sum()
    peak = by_hour.sort_values(['artikel', 'jam'], ascending=[False, True]).iloc[0]
    hour_insight = (
        f"Jam tercatat dengan artikel terbanyak adalah {int(peak['jam']):02d}.00 "
        f"({int(peak['artikel']):,} artikel; {100 * peak['artikel'] / known_time_rows:.2f}% "
        f"dari {known_time_rows:,} artikel dengan informasi jam). "
        f"Sebanyak {total - known_time_rows:,} artikel tanpa jam dikeluarkan dari heatmap. "
        "Waktu tanpa offset masih bergantung pada asumsi WIB (grafik 5)."
    )
else:
    hour_insight = (
        f"Seluruh {total:,} artikel tidak memiliki jam eksplisit. "
        "Puncak jam publikasi tidak dapat disimpulkan (grafik 5)."
    )

if math.isfinite(volume_length_r):
    direction = 'positif' if volume_length_r > 0 else 'negatif' if volume_length_r < 0 else 'nol'
    magnitude_note = (
        "Nilainya dekat nol, sehingga volume harian saja tidak menunjukkan hubungan linear "
        "yang kuat dengan panjang artikel. "
        if abs(volume_length_r) < 0.1 else ""
    )
    correlation_insight = (
        f"Korelasi kunci: Pearson r volume harian dengan rata-rata panjang artikel adalah "
        f"{volume_length_r:.3f} ({direction}; {len(correlation_days)} hari berartikel). "
        f"{magnitude_note}Koefisien ini menunjukkan asosiasi linear pada tingkat hari; "
        "tidak membuktikan bahwa beban publikasi mengubah panjang artikel. "
        "Periksa perubahan komposisi sumber sebelum membuat keputusan kapasitas (grafik 3 dan 6)."
    )
else:
    correlation_insight = (
        "Korelasi volume harian dan panjang artikel tidak terdefinisi karena variasi "
        "atau observasi tidak mencukupi; tidak dapat ditarik kesimpulan hubungan (grafik 3)."
    )

insights = [
    f"Tren waktu: {most_month['label_bulan']} mempunyai volume tertinggi "
    f"{int(most_month['artikel']):,} artikel ({most_month['persentase']:.2f}% korpus), "
    f"sedangkan {least_month['label_bulan']} terendah {int(least_month['artikel']):,}. "
    f"Bulan volume tertinggi mencatat {most_month['rata_per_hari']:.2f} artikel/hari kalender. "
    "Perbedaan ini belum menjelaskan penyebab perubahan; periksa kontribusi sumber dan "
    "cakupan pengumpulan sebelum mengaitkannya dengan peristiwa (grafik 1, 7 dan 9).",
    f"Distribusi dan anomali: rata-rata panjang {length['mean_words']:.2f} kata melebihi "
    f"median {length['median_words']:.2f}; P99 {length['p99']:.2f} kata. "
    f"Sebanyak {int(length['outliers']):,} artikel ({100 * length['outliers'] / total:.2f}%) "
    "ditandai IQR dan tetap disertakan. Median melengkapi rata-rata agar ringkasan panjang "
    "tidak hanya dipengaruhi ekor distribusi; flag tidak otomatis berarti salah (grafik 2 dan 10).",
    correlation_insight,
    f"Perilaku segmen: {len(sources)} sumber tercatat. {top_source['sumber']} terbesar "
    f"({top_source['persentase']:.2f}%), dan lima sumber teratas {top5_share:.2f}% korpus. "
    f"Rata-rata panjang per sumber berkisar {shortest_source['rata_kata']:.2f} kata "
    f"({shortest_source['sumber']}) sampai {longest_source['rata_kata']:.2f} "
    f"({longest_source['sumber']}). Analisis gabungan perlu didampingi hasil per sumber "
    "untuk mengurangi dominasi sumber dengan volume besar (grafik 4, 6 dan 9).",
    f"Rekomendasi praktis: jadikan rata-rata hari kerja {weekday_rate:.2f} artikel/hari "
    f"dan akhir pekan {weekend_rate:.2f} sebagai baseline historis kapasitas pengolahan, "
    f"dengan pemantauan khusus {busiest_day['hari']} ({busiest_day['rata_per_hari']:.2f}/hari). "
    f"Audit sampel dari {int(length['outliers']):,} flag outlier, pantau coverage setiap sumber, "
    "dan validasi baseline menggunakan data terbaru sebelum penjadwalan operasional. "
    "Pola arsip 2017 tidak cukup untuk menetapkan jadwal saat ini (grafik 2, 6 dan 8).",
    f"Pola musiman mingguan: {busiest_day['hari']} mempunyai rata-rata tertinggi "
    f"({busiest_day['rata_per_hari']:.2f} artikel/hari kalender). Senin–Jumat "
    f"{weekday_rate:.2f}, dibanding Sabtu–Minggu {weekend_rate:.2f}. "
    "Denominator memperhitungkan perbedaan jumlah kejadian hari, termasuk hari kosong "
    "dalam arsip. Pola mingguan ini tidak membuktikan musim antartahun (grafik 5 dan 8).",
    'Pola jam: ' + hour_insight,
    f"Kualitas data: {CLEANING['rows_raw']:,} artikel mentah menghasilkan {total:,} artikel bersih "
    f"({CLEANING['retained_percent']:.2f}% dipertahankan), setelah mengarantina "
    f"{CLEANING['rows_rejected']:,} dan menghapus {CLEANING['duplicates_removed']:,} "
    "salinan duplikat. Jumlah baris direkonsiliasi; checksum hasil M2 cocok. "
    "Angka ini merangkum metadata cleaning, bukan proses cleaning baru di M3.",
    f"Cakupan kalender: artikel tersedia pada {coverage_days} dari {len(daily)} hari "
    f"({100 * coverage_days / len(daily):.2f}%), dengan {empty_days} hari kosong. "
    "Coverage tanggal lengkap tidak menjamin seluruh artikel tiap media terkumpul; "
    "tetap periksa coverage per sumber (grafik 1 dan 9).",
]
report = '# Insight Analisis Berita Indonesia 2017\n\n'
report += f'Artikel bersih: {total:,}. Input SHA-256: `{CLEANING["clean_parquet_sha256"]}`.\n\n'
report += '\n\n'.join(f'{i}. **{text.split(":", 1)[0]}:**{text.split(":", 1)[1]}'
                        for i, text in enumerate(insights, 1))
report += '\n\n## Visualisasi Interaktif\n\n'
report += '\n'.join(f'- [{title}](figures/{name}.html)' for name, title in figures.items())
report += (
    '\n\nHTML dihasilkan secara lokal saat notebook dijalankan dan tidak disimpan di Git. '
    'Output Plotly tersimpan di notebook; buka di Jupyter untuk interaksi.\n'
    '\n## Batas Interpretasi\n\n'
    'Unit analisis adalah artikel dalam arsip, bukan peristiwa atau sensus publikasi Indonesia. '
    'Korelasi menggunakan agregat hari, bersifat deskriptif, dan tidak menyimpulkan sebab-akibat. '
    'Tidak tersedia koordinat terverifikasi. Timestamp tanpa offset diasumsikan WIB. '
    'Median/persentil memakai seluruh korpus bersih, sedangkan ambang IQR berasal dari artikel '
    'valid sebelum deduplikasi. Baseline 2017 perlu divalidasi sebelum diterapkan saat ini.\n'
)
(PROJECT_ROOT / 'output/insights.md').write_text(report, encoding='utf-8')
validation = {
    'daily_total_matches': int(daily['artikel'].sum()) == total,
    'monthly_total_matches': int(monthly['artikel'].sum()) == total,
    'source_total_matches': int(sources['artikel'].sum()) == total,
    'histogram_total_matches': int(histogram['artikel'].sum()) == total,
    'weekday_total_matches': int(weekday['artikel'].sum()) == total,
    'monthly_length_total_matches': int(length_monthly['artikel'].sum()) == total,
    'hourly_total_matches': known_time_rows + int(CLEANING['date_only_rows']) == total,
    'calendar_complete': len(daily) == 365 and daily['tanggal'].is_unique,
    'rolling_mean_matches': bool(np.allclose(daily['ma7'], daily['artikel'].rolling(7, min_periods=1).mean())),
    'html_files_exist': all((FIGURE_DIR / f'{name}.html').is_file() for name in figures),
}
assert all(validation.values()), validation
assert len(figures) == 10 and len(insights) >= 5
write_json(PROJECT_ROOT / 'output/eda_summary.json', {
    'rows_analysed': total,
    'clean_parquet_sha256': CLEANING['clean_parquet_sha256'],
    'visualisations': [{'file': f'output/figures/{name}.html', 'title': title} for name, title in figures.items()],
    'insights': insights, 'calendar_days': len(daily), 'days_with_articles': coverage_days,
    'known_time_rows': known_time_rows,
    'daily_volume_length_pearson_r': volume_length_r if math.isfinite(volume_length_r) else None,
    'correlation_unit': 'day with articles', 'correlation_days': len(correlation_days),
    'length_statistics': {key: float(value) for key, value in length.items()},
    'validation': validation,
    'runtime': {'python': platform.python_version(), 'polars': pl.__version__,
                'duckdb': duckdb.__version__, 'plotly': plotly.__version__,
                'pandas': pd.__version__, 'numpy': np.__version__},
})
display(Markdown(report))
print(f'Validasi lulus: {total:,} artikel, {len(figures)} grafik, {len(insights)} insight.')
connection.close()

# Insight Analisis Berita Indonesia 2017

Artikel bersih: 455,360. Input SHA-256: `ba3c4d94dc63a31d19d102127fbb993e80d097959600bb3cc2aca33a2f4fcea6`.

1. **Tren waktu:** 2017-08 mempunyai volume tertinggi 46,745 artikel (10.27% korpus), sedangkan 2017-09 terendah 25,177. Bulan volume tertinggi mencatat 1507.90 artikel/hari kalender. Perbedaan ini belum menjelaskan penyebab perubahan; periksa kontribusi sumber dan cakupan pengumpulan sebelum mengaitkannya dengan peristiwa (grafik 1, 7 dan 9).

2. **Distribusi dan anomali:** rata-rata panjang 272.35 kata melebihi median 246.00; P99 804.00 kata. Sebanyak 20,799 artikel (4.57%) ditandai IQR dan tetap disertakan. Median melengkapi rata-rata agar ringkasan panjang tidak hanya dipengaruhi ekor distribusi; flag tidak otomatis berarti salah (grafik 2 dan 10).

3. **Korelasi kunci:** Pearson r volume harian dengan rata-rata panjang artikel adalah -0.089 (negatif; 365 hari berartikel). Nilainya dekat nol, sehingga volume harian saja tidak menunjukkan hubungan linear yang kuat dengan panjang artikel. Koefisien ini menunjukkan asosiasi linear pada tingkat hari; tidak membuktikan bahwa beban publikasi mengubah panjang artikel. Periksa perubahan komposisi sumber sebelum membuat keputusan kapasitas (grafik 3 dan 6).

4. **Perilaku segmen:** 6 sumber tercatat. tribunnews.com terbesar (28.40%), dan lima sumber teratas 97.03% korpus. Rata-rata panjang per sumber berkisar 248.90 kata (liputan6.com) sampai 297.33 (kompas.com). Analisis gabungan perlu didampingi hasil per sumber untuk mengurangi dominasi sumber dengan volume besar (grafik 4, 6 dan 9).

5. **Rekomendasi praktis:** jadikan rata-rata hari kerja 1445.87 artikel/hari dan akhir pekan 756.51 sebagai baseline historis kapasitas pengolahan, dengan pemantauan khusus Rabu (1518.33/hari). Audit sampel dari 20,799 flag outlier, pantau coverage setiap sumber, dan validasi baseline menggunakan data terbaru sebelum penjadwalan operasional. Pola arsip 2017 tidak cukup untuk menetapkan jadwal saat ini (grafik 2, 6 dan 8).

6. **Pola musiman mingguan:** Rabu mempunyai rata-rata tertinggi (1518.33 artikel/hari kalender). Senin–Jumat 1445.87, dibanding Sabtu–Minggu 756.51. Denominator memperhitungkan perbedaan jumlah kejadian hari, termasuk hari kosong dalam arsip. Pola mingguan ini tidak membuktikan musim antartahun (grafik 5 dan 8).

7. **Pola jam:** Jam tercatat dengan artikel terbanyak adalah 15.00 (33,759 artikel; 7.41% dari 455,360 artikel dengan informasi jam). Sebanyak 0 artikel tanpa jam dikeluarkan dari heatmap. Waktu tanpa offset masih bergantung pada asumsi WIB (grafik 5).

8. **Kualitas data:** 455,678 artikel mentah menghasilkan 455,360 artikel bersih (99.93% dipertahankan), setelah mengarantina 52 dan menghapus 266 salinan duplikat. Jumlah baris direkonsiliasi; checksum hasil M2 cocok. Angka ini merangkum metadata cleaning, bukan proses cleaning baru di M3.

9. **Cakupan kalender:** artikel tersedia pada 365 dari 365 hari (100.00%), dengan 0 hari kosong. Coverage tanggal lengkap tidak menjamin seluruh artikel tiap media terkumpul; tetap periksa coverage per sumber (grafik 1 dan 9).

## Visualisasi Interaktif

- [1. Jumlah Artikel Harian dan Rata-rata 7 Hari](figures/01_daily_trend.html)
- [2. Distribusi Jumlah Kata per Artikel](figures/02_article_length_distribution.html)
- [3. Korelasi Metrik Harian (365 hari berartikel)](figures/03_daily_correlation.html)
- [4. Komposisi Sumber Artikel Terbesar](figures/04_source_composition.html)
- [5. Siklus Hari × Jam (455,360 artikel dengan jam)](figures/05_weekday_hour_heatmap.html)
- [6. Volume, Panjang Teks, dan Outlier Menurut Sumber](figures/06_source_multidimensional.html)
- [7. Jumlah Artikel per Bulan](figures/07_monthly_volume.html)
- [8. Intensitas Artikel Menurut Hari dalam Minggu](figures/08_weekday_pattern.html)
- [9. Volume Bulanan Lima Sumber Terbesar](figures/09_source_monthly_trend.html)
- [10. Rata-rata dan Median Panjang Artikel per Bulan](figures/10_monthly_article_length.html)

HTML dihasilkan secara lokal saat notebook dijalankan dan tidak disimpan di Git. Output Plotly tersimpan di notebook; buka di Jupyter untuk interaksi.

## Batas Interpretasi

Unit analisis adalah artikel dalam arsip, bukan peristiwa atau sensus publikasi Indonesia. Korelasi menggunakan agregat hari, bersifat deskriptif, dan tidak menyimpulkan sebab-akibat. Tidak tersedia koordinat terverifikasi. Timestamp tanpa offset diasumsikan WIB. Median/persentil memakai seluruh korpus bersih, sedangkan ambang IQR berasal dari artikel valid sebelum deduplikasi. Baseline 2017 perlu divalidasi sebelum diterapkan saat ini.


Validasi lulus: 455,360 artikel, 10 grafik, 9 insight.


### AI Disclosure Statement

ChatGPT/Codex membantu adaptasi template, implementasi temporal/korelasi,
visualisasi, perumusan insight berbasis agregasi, dan dokumentasi. Notebook ini
dijalankan ulang pada seluruh data bersih; checksum M2, rekonsiliasi agregat,
window tujuh hari, dan berkas grafik diperiksa. Mahasiswa bertanggung jawab
memahami kode, menilai interpretasi, dan memeriksa ulang temuan.